## 1 — Config & Imports

In [ ]:
import os, math, random, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

BASE_DIR = "/Users/jaswanthimandalapu/Documents/Sem2/CMPE 256 (Recommender Systems)/Final Project"
DATA_DIR = os.path.join(BASE_DIR, "llm_rec_data")   # full 1.9M user dataset
CKPT_PATH = os.path.join(BASE_DIR, "sasrec_best.pt")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# ── Model ────────────────────────────────────────────────────
MAX_LEN      = 50
HIDDEN       = 64
N_HEADS      = 2     # 64 / 2 = 32 per head
N_LAYERS     = 2
DROPOUT      = 0.2

# ── Training ─────────────────────────────────────────────────
BATCH_SIZE   = 1024
EPOCHS       = 50
LR           = 1e-3
WEIGHT_DECAY = 1e-2
PATIENCE     = 10

# ── Evaluation ───────────────────────────────────────────────
N_NEG_EVAL   = 100   # negatives sampled per user at evaluation
EVAL_BATCH   = 2048  # users per eval forward pass
KS           = [1, 5, 10, 20, 100]

# ── Tokens ───────────────────────────────────────────────────
PAD    = 0
OFFSET = 1   # item_idx i → vocab id  i + 1

## 2 — Data Loading

In [ ]:
COLS = ["user_idx", "item_idx", "timestamp"]

print("Loading parquet files...")
train_df = pd.read_parquet(os.path.join(DATA_DIR, "core", "interactions_train.parquet"), columns=COLS)
valid_df = pd.read_parquet(os.path.join(DATA_DIR, "core", "interactions_valid.parquet"), columns=COLS)
test_df  = pd.read_parquet(os.path.join(DATA_DIR, "core", "interactions_test.parquet"),  columns=COLS)
item_map = pd.read_parquet(os.path.join(DATA_DIR, "core", "item_id_mapping.parquet"))

n_items  = len(item_map)              # 310,334
n_users  = train_df["user_idx"].nunique()
vocab_size = n_items + OFFSET         # 0=PAD, 1..n_items=items

print(f"Users      : {n_users:,}")
print(f"Items      : {n_items:,}")
print(f"Train rows : {len(train_df):,}")
print(f"Valid rows : {len(valid_df):,}")
print(f"Test rows  : {len(test_df):,}")

## 3 — Build User Sequences

In [ ]:
print("Building user sequences...")

# Chronological item_idx list per user — stored as int32 numpy arrays to save RAM.
# 1.9M users × avg 8 items × 4 bytes ≈ 60 MB vs ~500 MB for Python lists.
train_seqs = (
    train_df.sort_values("timestamp")
    .groupby("user_idx")["item_idx"]
    .apply(lambda x: x.to_numpy(dtype=np.int32))
    .to_dict()
)

valid_targets = valid_df.set_index("user_idx")["item_idx"].to_dict()
test_targets  = test_df.set_index("user_idx")["item_idx"].to_dict()

# Test input = train history + valid item
test_seqs = {
    u: np.append(train_seqs[u], valid_targets[u]).astype(np.int32)
    for u in test_targets
}

lengths = [len(s) for s in train_seqs.values()]
print(f"Train seq lengths: min={min(lengths)}  max={max(lengths)}  "
      f"mean={np.mean(lengths):.1f}  median={np.median(lengths):.0f}")
print(f"Sequences built: {len(train_seqs):,} users")

## 4 — Dataset

In [ ]:
class SASRecDataset(Dataset):
    """
    For each user, builds:
      input  : train_seq[:-1]  shifted right by 1 (left-padded to MAX_LEN)
      pos    : train_seq[1:]   the next item at every position
      neg    : one random item per position (BPR negative)

    At position i the model sees items 0..i and predicts item i+1.
    Loss is only computed where pos != PAD.
    """

    def __init__(self, user_seqs: dict, max_len: int, n_items: int):
        self.seqs    = list(user_seqs.values())
        self.max_len = max_len
        self.n_items = n_items

    def __len__(self):
        return len(self.seqs)

    def __getitem__(self, idx):
        seq = self.seqs[idx]

        # input = all but last item, target = all but first (shifted by 1)
        inp = seq[:-1][-self.max_len:]
        tgt = seq[1:][-self.max_len:]

        inp_ids = (inp + OFFSET).tolist()
        tgt_ids = (tgt + OFFSET).tolist()
        neg_ids = [random.randint(1, self.n_items) for _ in tgt_ids]

        # Left-pad to max_len
        pad     = self.max_len - len(inp_ids)
        inp_ids = [PAD] * pad + inp_ids
        tgt_ids = [PAD] * pad + tgt_ids
        neg_ids = [PAD] * pad + neg_ids

        return (torch.tensor(inp_ids, dtype=torch.long),
                torch.tensor(tgt_ids, dtype=torch.long),
                torch.tensor(neg_ids, dtype=torch.long))

## 5 — SASRec Model

In [ ]:
class SASRec(nn.Module):
    """
    Sequential Attention-based Recommendation.

    Key difference from BERT4Rec:
    - Causal (left-to-right) attention: position i only attends to 0..i
    - No [MASK] token — at eval, the last position naturally predicts the next item
    - BPR loss instead of cross-entropy: O(hidden) per step, not O(vocab)
    """

    def __init__(self, vocab_size, hidden, n_heads, n_layers, max_len, dropout):
        super().__init__()
        self.item_emb = nn.Embedding(vocab_size, hidden, padding_idx=PAD)
        self.pos_emb  = nn.Embedding(max_len, hidden)
        self.emb_norm = nn.LayerNorm(hidden)
        self.emb_drop = nn.Dropout(dropout)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=hidden, nhead=n_heads,
            dim_feedforward=hidden * 4,
            dropout=dropout, activation="gelu",
            batch_first=True, norm_first=True,
        )
        self.encoder  = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.out_norm = nn.LayerNorm(hidden)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, std=0.02)
            elif isinstance(m, nn.LayerNorm):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)

    def _causal_mask(self, L, device):
        # Upper-triangular float mask: -inf where attention is blocked
        m = torch.zeros(L, L, device=device)
        m.fill_(float("-inf"))
        return torch.triu(m, diagonal=1)

    def encode(self, x):
        """x : (B, L) → (B, L, hidden)  hidden states at every position."""
        B, L = x.shape
        pos      = torch.arange(L, device=x.device).unsqueeze(0).expand(B, -1)
        pad_mask = (x == PAD)
        attn_mask = self._causal_mask(L, x.device)

        h = self.emb_drop(self.emb_norm(self.item_emb(x) + self.pos_emb(pos)))
        h = self.encoder(h, mask=attn_mask, src_key_padding_mask=pad_mask)
        return self.out_norm(h)   # (B, L, hidden)

    def score(self, h, item_ids):
        """Dot-product score between hidden states and item embeddings.
        h        : (B, hidden)
        item_ids : (B, K)  — K candidates per user
        returns  : (B, K)
        """
        emb = self.item_emb(item_ids)          # (B, K, hidden)
        return (h.unsqueeze(1) * emb).sum(-1)  # (B, K)


model = SASRec(vocab_size, HIDDEN, N_HEADS, N_LAYERS, MAX_LEN, DROPOUT).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"SASRec  |  vocab={vocab_size:,}  hidden={HIDDEN}  layers={N_LAYERS}  heads={N_HEADS}")
print(f"Parameters: {n_params:,}")

## 6 — Training Infrastructure

In [ ]:
train_ds = SASRecDataset(train_seqs, MAX_LEN, n_items)
train_dl = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=4, pin_memory=(DEVICE.type == "cuda"),
    persistent_workers=True,
)

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR * 0.01)

print(f"Training users   : {len(train_ds):,}")
print(f"Batches per epoch: {len(train_dl):,}")


def bpr_loss(pos_scores, neg_scores, mask):
    """Bayesian Personalised Ranking loss over valid (non-pad) positions."""
    loss = -torch.log(torch.sigmoid(pos_scores - neg_scores) + 1e-8)
    return (loss * mask).sum() / mask.sum()


def train_one_epoch(model, dl):
    model.train()
    total_loss, n_batches = 0.0, 0

    for inp, pos, neg in dl:
        inp, pos, neg = inp.to(DEVICE), pos.to(DEVICE), neg.to(DEVICE)
        mask = (pos != PAD).float()          # (B, L) 1 where valid

        h          = model.encode(inp)       # (B, L, hidden)
        pos_scores = model.score(h.view(-1, HIDDEN),
                                 pos.view(-1, 1)).squeeze(-1).view(inp.shape)
        neg_scores = model.score(h.view(-1, HIDDEN),
                                 neg.view(-1, 1)).squeeze(-1).view(inp.shape)

        loss = bpr_loss(pos_scores, neg_scores, mask)
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()
        n_batches  += 1

    return total_loss / n_batches

## 7 — Evaluation (100 Sampled Negatives)

In [ ]:
def _build_eval_input(user_seqs: dict, max_len: int):
    """Left-padded input matrix + user order list for all eval users."""
    rows, order = [], []
    for u, seq in user_seqs.items():
        tokens  = (seq[-max_len:] + OFFSET).tolist()
        pad     = max_len - len(tokens)
        tokens  = [PAD] * pad + tokens
        rows.append(tokens)
        order.append(u)
    return torch.tensor(rows, dtype=torch.long), order


@torch.no_grad()
def evaluate(model, user_seqs: dict, targets: dict, n_items: int, ks=KS):
    """
    100-negative sampled evaluation.

    For each user:
      1. Encode their history, take the hidden state at the last position.
      2. Score the 1 positive + 100 random negatives (101 candidates total).
      3. Rank the positive among 101 candidates → Hit@K, NDCG@K.
    """
    model.eval()
    all_input, user_order = _build_eval_input(user_seqs, MAX_LEN)

    hits  = {k: 0   for k in ks}
    ndcgs = {k: 0.0 for k in ks}
    n_eval = 0

    for start in range(0, len(user_order), EVAL_BATCH):
        end         = start + EVAL_BATCH
        batch_in    = all_input[start:end].to(DEVICE)     # (B, L)
        batch_users = user_order[start:end]
        B           = batch_in.shape[0]

        h_last = model.encode(batch_in)[:, -1, :]         # (B, hidden)

        # Build candidate tensor: column 0 = positive, columns 1-100 = negatives
        candidates = torch.randint(1, n_items + 1, (B, N_NEG_EVAL), device=DEVICE)
        pos_ids    = torch.tensor(
            [targets[u] + OFFSET for u in batch_users], device=DEVICE
        ).unsqueeze(1)                                     # (B, 1)
        candidates = torch.cat([pos_ids, candidates], dim=1)  # (B, 101)

        scores = model.score(h_last, candidates)           # (B, 101)
        # Rank of the positive (index 0) among all 101 candidates
        ranks  = (scores[:, 1:] >= scores[:, :1]).sum(dim=1) + 1  # (B,) 1-indexed

        for rank in ranks.tolist():
            for k in ks:
                if rank <= k:
                    hits[k]  += 1
                    ndcgs[k] += 1.0 / math.log2(rank + 1)
            n_eval += 1

    metrics = {}
    for k in ks:
        metrics[f"Hit@{k}"]  = hits[k]  / n_eval
        metrics[f"NDCG@{k}"] = ndcgs[k] / n_eval
    return metrics, n_eval

## 8 — Training Loop

In [ ]:
best_ndcg10   = 0.0
best_state    = None
patience_left = PATIENCE
history       = []
t0            = time.time()

for epoch in range(1, EPOCHS + 1):
    loss = train_one_epoch(model, train_dl)
    scheduler.step()

    val_metrics, n_val = evaluate(model, train_seqs, valid_targets, n_items)
    ndcg10 = val_metrics["NDCG@10"]

    history.append({"epoch": epoch, "train_loss": round(loss, 4),
                    **{k: round(v, 4) for k, v in val_metrics.items()}})

    improved = ndcg10 > best_ndcg10
    if improved:
        best_ndcg10   = ndcg10
        best_state    = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        patience_left = PATIENCE
        torch.save(best_state, CKPT_PATH)
    else:
        patience_left -= 1

    elapsed = time.time() - t0
    marker  = " ★" if improved else ""
    print(f"Epoch {epoch:3d}/{EPOCHS}  loss={loss:.4f}  "
          f"Hit@10={val_metrics['Hit@10']:.4f}  NDCG@10={ndcg10:.4f}  "
          f"[best={best_ndcg10:.4f}]  {elapsed/60:.1f}min{marker}")

    if patience_left == 0:
        print(f"Early stopping at epoch {epoch}.")
        break

print(f"\nBest val NDCG@10 = {best_ndcg10:.4f}")

## 9 — Final Evaluation

In [ ]:
model.load_state_dict(best_state or torch.load(CKPT_PATH, map_location=DEVICE))

val_metrics,  n_val  = evaluate(model, train_seqs, valid_targets, n_items)
test_metrics, n_test = evaluate(model, test_seqs,  test_targets,  n_items)

results = pd.DataFrame({"Validation": val_metrics, "Test": test_metrics})
results.index.name = "Metric"

print("=" * 60)
print(f"  SASRec Results  |  users={n_test:,}  negatives={N_NEG_EVAL}")
print("=" * 60)
display(results.style.format("{:.4f}").set_caption(
    f"Hit@K and NDCG@K — {N_NEG_EVAL} sampled negatives"))

## 10 — Training Curve

In [ ]:
import matplotlib.pyplot as plt

hist_df = pd.DataFrame(history)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(hist_df["epoch"], hist_df["train_loss"], marker="o", markersize=3)
axes[0].set_title("BPR Training Loss")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss"); axes[0].grid(True, alpha=0.3)

for k in [1, 5, 10, 20]:
    axes[1].plot(hist_df["epoch"], hist_df[f"Hit@{k}"], marker="o", markersize=3, label=f"Hit@{k}")
axes[1].set_title("Hit@K — Validation"); axes[1].set_xlabel("Epoch")
axes[1].legend(); axes[1].grid(True, alpha=0.3)

for k in [1, 5, 10, 20]:
    axes[2].plot(hist_df["epoch"], hist_df[f"NDCG@{k}"], marker="o", markersize=3, label=f"NDCG@{k}")
axes[2].set_title("NDCG@K — Validation"); axes[2].set_xlabel("Epoch")
axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, "sasrec_training_curve.png"), dpi=150, bbox_inches="tight")
plt.show()